# Análisis de consumo para evadir contraseña (HARDWARE)

---
**Este no es el notebook completo, ver el fichero con (PRINCIPAL) en el nombre.**

---

Primero tenemos que seleccionar nuestro hardware, estableciendo las variables `SCOPETYPE` y `PLATFORM` al valor `'CWNANO'`.

In [ ]:
SCOPETYPE = 'CWNANO'
PLATFORM = 'CWNANO'
SS_VER = 'SS_VER_2_1'

Este código conectará el osciloscopio y realizará una configuración básica. Ahora simplemente utilizaremos un script de configuración especial para hacer esto. Este script contiene los comandos que ejecutamos por separado anteriormente.

In [ ]:
%run "../chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb"

El siguiente código compila el firmware para el objetivo.

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd Firmwares/basic-passwdcheck
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

Por último, lo único que queda es programar el dispositivo, lo cual se hace con la siguiente línea:

In [ ]:
cw.program_target(scope, prog, "Firmwares/basic-passwdcheck/basic-passwdcheck-{}.hex".format(PLATFORM))

Para facilitar la interacción con el hardware, definimos una función que prueba una contraseña y devuelve una traza de potencia.

El propósito de esta función es automatizar una prueba de contraseña completa contra el microcontrolador víctima y capturar la traza de consumo de energía eléctrica producida durante la verificación.

Para lograrlo de forma reproducible:

1. Reinicia el microcontrolador objetivo (el target) a un estado limpio inicial, ya que en el firmware basic-passwdcheck.c, tras fallar o acertar una contraseña, el  microcontrolador entra deliberadamente en un bucle infinito de bloqueo (`while(1);`). Reiniciar el microcontrolador es la única manera de devolverlo al inicio de main().
2. Espera a que el target termine de arrancar vaciando del buffer serie (`while num_char > 0`) todos los mensajes de bienvenida que este envía por UART, garantizando que el microcontrolador ha completado su rutina de arranque y se encuentra bloqueado esperando la entrada en `my_read(...)`.
3. Arma el osciloscopio de ChipWhisperer (`scope.arm()`) para que quede a la escucha del pin de disparo (trigger).
4. Envía el intento de contraseña por puerto serie (`target.write(pass_guess)`).
5. Captura las muestras (`scope.capture()`) del conversor analógico-digital (ADC) mientras el microcontrolador ejecuta la comparación en C (`correct_passwd[i] != passwd[i]`).
6. Devuelve la traza en forma de array de NumPy, normalizado en coma flotante en el rango [-0.5, 0.5], para su posterior análisis o representación gráfica (`scope.get_last_trace()`).

In [ ]:
def cap_pass_trace(pass_guess):
    reset_target(scope)
    num_char = target.in_waiting()
    while num_char > 0:
        target.read(num_char, 10)
        time.sleep(0.01)
        num_char = target.in_waiting()

    scope.arm()
    target.write(pass_guess)
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')

    trace = scope.get_last_trace()
    return trace

Tampoco necesitamos las 5000 muestras que vienen por defecto en la traza. 3000 es un buen punto de partida para la mayoría de los objetivos:

In [ ]:
scope.adc.samples = 3000

---
**Agradecimientos / Referencias:**  
Este material está basado y adaptado a partir del notebook original [*Lab 2_1B - Power Analysis for Password Bypass (HARDWARE).ipynb*](https://github.com/newaetech/chipwhisperer-jupyter/blob/main/courses/sca101/Lab%202_1B%20-%20Power%20Analysis%20for%20Password%20Bypass%20(HARDWARE).ipynb) del repositorio oficial de [NewAE Technology](https://github.com/newaetech/chipwhisperer-jupyter).